Machine Learning Zoomcamp 2026 — Homework 4

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Load the official 2026 dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nMissing values before preparation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Prepare missing values
for col in df.columns:
    if col == "converted":
        continue
    if df[col].dtype == "object" or isinstance(df[col].dtype, pd.CategoricalDtype):
        df[col] = df[col].fillna("NA")
    else:
        df[col] = df[col].fillna(0.0)

# 3. Split the data exactly as instructed
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train["converted"].values
y_val = df_val["converted"].values

# Features and target
def get_features(data):
    return data.drop(columns=["converted"])

# Train a DictVectorizer and Logistic Regression model
def train_model(train_data, val_data, C=1.0):
    X_train = get_features(train_data)
    X_val = get_features(val_data)

    dv = DictVectorizer(sparse=True)
    train_dicts = X_train.to_dict(orient="records")
    val_dicts = X_val.to_dict(orient="records")

    X_train_encoded = dv.fit_transform(train_dicts)
    X_val_encoded = dv.transform(val_dicts)

    model = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000
    )
    model.fit(X_train_encoded, train_data["converted"].values)

    probabilities = model.predict_proba(X_val_encoded)[:, 1]
    return dv, model, probabilities

# 4. Question 1: Numerical feature AUC
print("\nQUESTION 1: NUMERICAL FEATURE AUC")
numeric_features = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]

for feature in numeric_features:
    score = roc_auc_score(y_train, df_train[feature])
    if score < 0.5:
        score = roc_auc_score(y_train, -df_train[feature])
    print(f"{feature}: {score:.3f}")

# 5. Question 2: Validation AUC
dv, model, val_probs = train_model(df_train, df_val, C=1.0)
auc_val = roc_auc_score(y_val, val_probs)
print("\nQUESTION 2: Validation AUC =", round(auc_val, 3))

# 6. Question 3: Precision and recall intersection
thresholds = np.arange(0.0, 1.001, 0.01)
pr_results = []

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(int)
    precision = precision_score(
        y_val, predictions, zero_division=0
    )
    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    # Ignore thresholds where both metrics are zero
    if precision == 0 and recall == 0:
        continue

    pr_results.append(
        (threshold, precision, recall, abs(precision - recall))
    )

best_intersection = min(pr_results, key=lambda row: row[3])
print("\nQUESTION 3: Precision/recall intersection")
print("Threshold:", round(best_intersection[0], 2))
print("Precision:", round(best_intersection[1], 3))
print("Recall:", round(best_intersection[2], 3))

# 7. Question 4: Threshold with maximum F1
f1_results = []

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(int)
    f1 = f1_score(y_val, predictions, zero_division=0)
    f1_results.append((threshold, f1))

best_f1 = max(f1_results, key=lambda row: row[1])
print("\nQUESTION 4: Maximum F1")
print("Threshold:", round(best_f1[0], 2))
print("F1 score:", round(best_f1[1], 3))

# 8. Question 5: Five-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = []

for train_idx, val_idx in kf.split(df_full_train):
    fold_train = df_full_train.iloc[train_idx]
    fold_val = df_full_train.iloc[val_idx]

    _, _, fold_probs = train_model(fold_train, fold_val, C=1.0)
    fold_auc = roc_auc_score(fold_val["converted"].values, fold_probs)
    cv_scores.append(fold_auc)

print("\nQUESTION 5: Five-fold CV")
print("Fold AUCs:", [round(s, 3) for s in cv_scores])
print("Mean AUC:", round(np.mean(cv_scores), 3))
print("Standard deviation:", round(np.std(cv_scores), 3))

# 9. Question 6: Tune C using the same five folds
print("\nQUESTION 6: Hyperparameter tuning")
tuning_results = []

for C in [0.000001, 0.001, 1]:
    scores = []

    for train_idx, val_idx in kf.split(df_full_train):
        fold_train = df_full_train.iloc[train_idx]
        fold_val = df_full_train.iloc[val_idx]

        _, _, fold_probs = train_model(fold_train, fold_val, C=C)
        fold_auc = roc_auc_score(
            fold_val["converted"].values, fold_probs
        )
        scores.append(fold_auc)

    mean_auc = np.mean(scores)
    std_auc = np.std(scores)

    tuning_results.append((C, mean_auc, std_auc))
    print(
        f"C={C}: mean AUC={mean_auc:.3f}, "
        f"std={std_auc:.3f}"
    )

# Highest mean AUC wins; ties go to lower std, then smaller C
tuning_results.sort(key=lambda x: (-x[1], x[2], x[0]))
print("\nBest C:", tuning_results[0][0])


Dataset shape: (5000, 9)

Missing values before preparation:
lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64

QUESTION 1: NUMERICAL FEATURE AUC
lead_score: 0.788
number_of_courses_viewed: 0.723
interaction_count: 0.765
annual_income: 0.608

QUESTION 2: Validation AUC = 0.732

QUESTION 3: Precision/recall intersection
Threshold: 0.63
Precision: 0.733
Recall: 0.725

QUESTION 4: Maximum F1
Threshold: 0.41
F1 score: 0.758

QUESTION 5: Five-fold CV
Fold AUCs: [np.float64(0.731), np.float64(0.73), np.float64(0.736), np.float64(0.741), np.float64(0.721)]
Mean AUC: 0.732
Standard deviation: 0.007

QUESTION 6: Hyperparameter tuning
C=1e-06: mean AUC=0.617, std=0.019
C=0.001: mean AUC=0.749, std=0.010
C=1: mean AUC=0.732, std=0.007

Best C: 0.001


Machine Learning Zoomcamp 2026 — Homework 3

In [2]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

# 1. Load the official 2026 dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)

# 2. Question 1: Mode of industry
print("\nQUESTION 1: Industry mode")
print("Mode:", df["industry"].mode().iloc[0])

# 3. Prepare features and fill missing values
target = df["converted"].copy()
X = df.drop(columns=["converted"]).copy()

categorical = X.select_dtypes(include=["object"]).columns.tolist()
numerical = X.select_dtypes(exclude=["object"]).columns.tolist()

X[categorical] = X[categorical].fillna("NA")
X[numerical] = X[numerical].fillna(0.0)

# 4. Question 2: Correlations
print("\nQUESTION 2: Correlation")

corr = X[numerical].corr().abs()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for a, b in pairs:
    print(f"{a} and {b}: {corr.loc[a, b]:.4f}")

best_pair = max(pairs, key=lambda p: corr.loc[p[0], p[1]])
print("Highest correlation pair:", best_pair)

# 5. Split using the exact Homework 3 settings
data = X.copy()
data["converted"] = target

df_full_train, df_test = train_test_split(
    data, test_size=0.2, random_state=42
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train.pop("converted")
y_val = df_val.pop("converted")

# 6. Question 3: Mutual information
print("\nQUESTION 3: Mutual information")

mi_scores = {}
for col in categorical:
    mi_scores[col] = mutual_info_score(y_train, df_train[col])

for col, score in sorted(
    mi_scores.items(), key=lambda item: item[1], reverse=True
):
    print(f"{col}: {score:.4f} (rounded: {round(score, 2):.2f})")

print("Highest MI feature:", max(mi_scores, key=mi_scores.get))

# Helper to train logistic regression with one-hot encoding
def fit_and_predict(train_x, validation_x, C=1.0):
    dv = DictVectorizer(sparse=False)

    train_matrix = dv.fit_transform(
        train_x.to_dict(orient="records")
    )
    validation_matrix = dv.transform(
        validation_x.to_dict(orient="records")
    )

    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    )
    model.fit(train_matrix, y_train)

    return model.predict(validation_matrix)

# 7. Question 4: Validation accuracy
print("\nQUESTION 4: Accuracy")

predictions = fit_and_predict(df_train, df_val)
original_accuracy = accuracy_score(y_val, predictions)

print("Accuracy:", original_accuracy)
print("Rounded to 2 decimals:", round(original_accuracy, 2))

# 8. Question 5: Feature elimination
print("\nQUESTION 5: Feature elimination")

differences = {}

for col in [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]:
    reduced_train = df_train.drop(columns=[col])
    reduced_val = df_val.drop(columns=[col])

    reduced_predictions = fit_and_predict(
        reduced_train, reduced_val
    )
    reduced_accuracy = accuracy_score(y_val, reduced_predictions)

    difference = abs(original_accuracy - reduced_accuracy)
    differences[col] = difference

    print(
        f"Without {col}: accuracy={reduced_accuracy:.4f}, "
        f"absolute difference={difference:.4f}"
    )

print("Smallest difference feature:", min(differences, key=differences.get))

# 9. Question 6: Tune C
print("\nQUESTION 6: Parameter tuning")

c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}

for C in c_values:
    predictions = fit_and_predict(df_train, df_val, C=C)
    accuracy = accuracy_score(y_val, predictions)
    c_scores[C] = accuracy
    print(f"C={C}: accuracy={accuracy:.3f}")

best_C = max(c_values, key=lambda c: (c_scores[c], -c))
print("Best C:", best_C)


Dataset shape: (5000, 9)

QUESTION 1: Industry mode
Mode: technology

QUESTION 2: Correlation
interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228
Highest correlation pair: ('interaction_count', 'lead_score')

QUESTION 3: Mutual information
lead_source: 0.0343 (rounded: 0.03)
employment_status: 0.0190 (rounded: 0.02)
industry: 0.0020 (rounded: 0.00)
location: 0.0010 (rounded: 0.00)
Highest MI feature: lead_source

QUESTION 4: Accuracy
Accuracy: 0.645
Rounded to 2 decimals: 0.65

QUESTION 5: Feature elimination
Without lead_source: accuracy=0.6420, absolute difference=0.0030
Without number_of_courses_viewed: accuracy=0.6430, absolute difference=0.0020
Without interaction_count: accuracy=0.6010, absolute difference=0.0440
Smallest difference feature: number_of_courses_viewed

QUESTION 6: Parameter tuning
C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0